In [ ]:
# Generates event data for the probe responses
# The .tsv events files were bugged, so a separate .txt file storing the duplicate responses was used
# A new .tsv was generated for each subj

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import os

import datetime as dt

import re

from dotenv import load_dotenv

In [ ]:
# Set constants and paths

load_dotenv()  # Load environment variables from .env file
ROOT = os.getenv("ROOT")

RAW_DATA = ROOT

SUBJECTS = [f"{i:03d}" for i in range(1, 25)]  # '001' through '024'
SESSIONS = ["01", "02", "03"]

In [ ]:
def sync_eeg_behavior(text_path, tsv_path, output_path):
    # --- 1. Parse Text Notes ---
    text_blocks = []
    current_block = None
    probe_re = re.compile(r"MW question asked at time\s+([\d\.]+)")
    key_re = re.compile(r"Key\s+(?P<key>\d+).*?time\s+(?P<time>[\d\.]+)")

    with open(text_path, "r") as f:
        for line in f:
            p_match = probe_re.search(line)
            if p_match:
                if current_block:
                    text_blocks.append(current_block)
                current_block = {"p_time": float(p_match.group(1)), "resps": []}
            k_match = key_re.search(line)
            if k_match and current_block:
                current_block["resps"].append(
                    {"k": int(k_match.group("key")), "t": float(k_match.group("time"))}
                )
        if current_block:
            text_blocks.append(current_block)

    # --- 2. Load Spreadsheet ---
    ss = pd.read_csv(tsv_path, sep="\t")
    ss_probes = ss[ss["value"] == 128].copy()
    # ss_probes = ss[ss['value'] == 128]['onset'].tolist()

    # --- 3. Synchronize ---
    final_events = []
    val_map = {0: 0, 1: 2, 2: 4, 3: 8}
    expected_offset = ss_probes["onset"].tolist()[0] - text_blocks[0]["p_time"]
    tight_tol = 1.0
    wide_tol = 5.0

    for block in text_blocks:
        diffs = (ss_probes["onset"] - block["p_time"]) - expected_offset

        # first pass: tight tolerance
        candidates = ss_probes[diffs.abs() < tight_tol]

        # fallback: nearest within wider tolerance
        if candidates.empty and not ss_probes.empty:
            nearest_idx = diffs.abs().idxmin()
            if abs(diffs.loc[nearest_idx]) < wide_tol:
                candidates = ss_probes.loc[[nearest_idx]]

        if candidates.empty:
            print(f"[WARN] No probe match for text probe at {block['p_time']:.3f}s")
            continue

        actual_ss_time = candidates.iloc[0]["onset"]
        offset = actual_ss_time - block["p_time"]

        final_events.append([actual_ss_time, "stimulus", 128])
        for r in block["resps"]:
            final_events.append([r["t"] + offset, "response", val_map.get(r["k"], 99)])

    # --- 4. Export ---
    out_df = pd.DataFrame(final_events, columns=["onset", "trial_type", "value"])
    sfreq = 256
    out_df["duration"] = 0.0
    out_df["sample"] = (out_df["onset"] * sfreq).astype(int)

    # Reorder to match BIDS standard
    out_df = out_df[["onset", "duration", "trial_type", "sample", "value"]]
    out_df = out_df.sort_values("onset").reset_index(drop=True)
    out_df.to_csv(output_path, sep="\t", index=False)
    return out_df


for subject in SUBJECTS:
    for session in SESSIONS:
        tsv_path = f"{RAW_DATA}sub-{subject}/ses-{session}/eeg/sub-{subject}_ses-{session}_task-meditation_events.tsv"
        output_path = f"{RAW_DATA}sub-{subject}/ses-{session}/eeg/sub-{subject}_ses-{session}_task-meditation-fixed_events.tsv"
        if not os.path.exists(tsv_path):
            data_file_path = f"{RAW_DATA}sub-{subject}/ses-{session}/eeg/sub-{subject}_ses-{session}_task-meditation_eeg.bdf"
            # if the raw data doesn't have a corresponding .tsv there is an error
            # otherwise there just inst a session for that subj and thats fine
            if os.path.exists(data_file_path):
                print(f"[ERROR] events TSV missing: {tsv_path}")
            continue

        # do NOT mutate loop vars
        notes_subject = subject[1:]  # "003" -> "03"
        notes_session_num = session[1:]  # "02" -> "2"
        notes_session = "" if notes_session_num == "1" else f"_{notes_session_num}"

        notes_file = f"{RAW_DATA}code/MW_Current_TextFileBIDS/sub{notes_subject}{notes_session}_info.txt"

        if not os.path.exists(notes_file):
            print(f"[ERROR] notes file missing: {notes_file}")
            continue

        try:
            sync_eeg_behavior(notes_file, tsv_path, output_path)
            print(f"[INFO] Processed subject {subject}, session {session}")
        except Exception as e:
            print(
                f"[ERROR] Failed to process subject {subject}, session {session}: {e}"
            )